This blog explores how trading volume and the key corporate events surrounding Elon Musk's takeover of Twitter/X affected and predicted the next-day stock volatility from the IPO in 2013 to the day it was taken private in 2022. 

Twitter is one of the most used social media platforms in the world, currently under the name X after the Musk takeover. Whilst the actual figure is not available, multiple sources say that there are around 500 million monthly users on X as of 2026. Twitter first went public on the 7th of November 2013, at $26 per share and it is now private, allowing Musk to impose his strategy of free speech principles to the platform. Musk fired four of the top executives at Twitter after completing his acquisition (Gordon, 2022).

The analysis to answer the research question this blog uncovers uses two methods, OLS regression and a random forest model, to test whether trading volume and two of the most important events in Musk's takeover can predict how volatile the stock would be the following day. The two events will be explained later in the blog and the reason why those two were chosen. The data used is taken from Kaggle, and is the daily stock price data, including open, close, high, low, adjusted close and volume, covering the full period of Twitter as a public company.

In [11]:
#Imported all the libraries I will need at the beginning of my code to group them all together.

import pandas as pd #data manipulation
import numpy as np
import matplotlib.pyplot as plt #plotting graphs
import seaborn as sns #plotting graphs

import statsmodels.api as sm
import statsmodels.formula.api as smf #regression

from sklearn.ensemble import RandomForestRegressor #random forest
from sklearn.model_selection import train_test_split #splits data for the random forest
from sklearn.metrics import mean_squared_error, r2_score #measures accuracy of model

In [12]:
#Loading up the dataset, making sure it is in the same folder as the blog. 

df = pd.read_csv('../Data/twitter-stocks.csv')
df.head()

#Used .head() to see if it worked and how it looks.

FileNotFoundError: [Errno 2] No such file or directory: '../Data/twitter-stocks.csv'

In [ ]:
df.info()
df.describe()

#More data description and to see if I was dealing with missing data.

In [ ]:
#No missing data in my data set so I could move onto adding in the required variables to complete the regression.

#Converting the date from an object (as seen from .info() to datetime).
df['Date'] = pd.to_datetime(df['Date'])

#Organising by date, oldest first.
df = df.sort_values('Date').reset_index(drop=True)

#Creating daily return (% change in closing price).
df['Daily_Return'] = df['Close'].pct_change() * 100

#Creating volatility (absolute value of daily return).
df['Volatility'] = df['Daily_Return'].abs()

#Creating next-day volatility.
df['Next_Day_Volatility'] = df['Volatility'].shift(-1)

#Creating a 20 day moving average (to allow for better
df['MA20'] = df['Close'].rolling(window=20).mean()

#Flagging the Musk era (from January 31st 2022 onwards as this is when he first bought stock).
df['Musk_Era'] = (pd.to_datetime(df['Date']) >= '2022-1-31').astype(int)

#Drops the last row and the first 20, as it will have NaN for next day volatility and 20 day moving average.
df = df.dropna()

df.head()

#.head() will allow me to see the data again with the new columns.

In [ ]:
df.info()
df.head(2250) #seeing if the Musk Era does indeed get flagged.

You can see that there are 20 less rows now, because the first 19 days of stock price have been removed as there cannot be a 20-day moving average calculated, and the last day has been removed as there is no next day volatility.

The figure below shows the full-price history of Twitter stock from its IPO in November 2013 to the point where Elon Musk takes it private in October 2022, with the 20-day moving average overlaid. The moving average smooths out short-term fluctuations and makes the longer-term trend easier to read.

In [ ]:
#Figure 1: Full price history 2013–2022 with moving average plotted over it.
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], color='royalblue', linewidth=1)
plt.plot(df['Date'], df['MA20'], color='orange', linewidth=1, linestyle='--', label='20-day MA')
plt.title('Twitter Stock Price History (2013–2022)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

As you can see, Twitter stock was very volatile, long before Elon Musk was involved. Following the IPO, there is an initial surge in the stock, rising to over $70 per share, before falling, and eventually in 2016 it would reach prices as low as $20 per share. This is a result of stagnating user numbers and leadership issues that would last until 2018, with the greatest rise observed in 2020, due to the COVID-19 pandemic. Increased screen time meant social media stocks benefited. Before the stock gets taken private, you can see in 2022 it starts to rise again, as Musk begins buying Twitter stock.

Figure 2 focuses on the Twitter stock price in 2022, with 6 key dates relating to the Musk acquisition, marked as dotted, vertical lines. These events were found according to Birath (2022) and Wile (2022), telling the story of his takeover.

In [ ]:
#Figure 2: Show 2022 only, with the key Elon Musk dates overlayed.
df_2022 = df[df['Date'].dt.year == 2022]

#Musk dates
musk_events = {
    'Musk purchases Twitter stock': '2022-01-31',
    'Musk takes 9.2% stake': '2022-04-04',
    'Buyout offer ($54.20 PPS)': '2022-04-14',
    'Deal suspended': '2022-05-13',
    'Musk tries to exit': '2022-07-08',
    'Deal completed': '2022-10-27'
}

plt.figure(figsize=(14, 6))
plt.plot(df_2022['Date'], df_2022['Close'], color='royalblue', linewidth=1.5, label='Closing Price')
plt.plot(df_2022['Date'], df_2022['MA20'], color='orange', linewidth=1, linestyle='--', label='20-day MA')

colors = ['grey', 'red', 'green', 'purple', 'brown', 'black']
for (event, date), color in zip(musk_events.items(), colors):
    plt.axvline(pd.to_datetime(date), color=color, linestyle='--', linewidth=1.2, label=event)

plt.title('Twitter Stock Price During Musk Takeover (2022)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

Starting on January 31st, 2022, Musk begins purchasing Twitter stock, and it's not until April 4th that he announces he has 9.2% holding in Twitter stock, which legally he had to declare ten days after passing 5% according to the SEC but he failed to do so. On the 14th of April, Musk updates his SEC announcement to include his intent to purchase Twitter at a price of $54.20 per share, or about $44 billion. Twitter and Musk come to a merge deal on the 25th of April, however the deal is suspended on the 13th of May. Musk tries to exit the deal on the 8th of July, with Twitter filing a lawsuit in the following days. It is not until October 3rd when Musk announces he would complete the deal as originally agreed, and on the 27th of October, the deal is completed, Twitter is taken private and Musk begins his restructuring of Twitter.

The figure signifies the changes in the stock price following each of these events, with the market clearly being driven by Musk's actions and statements.

Figure 3 shows daily volatility, measured as the absolute value of the daily return (percentage change in closing price), across the full period. Volatility refers to the degree of price fluctuations over time, used to measure uncertainty and risk in stocks.

In [ ]:
#Figure 3: Volatility over time
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Volatility'], color='crimson', linewidth=0.8, label='Daily Volatility')
plt.title('Twitter Stock Volatility (2013–2022)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Absolute Daily Return (%)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

Volatility was relatively low and stable for most of Twitter's history, with occasional spikes around earning announcements and major news events. Low volatility reflects low, stable, gradual movement in price changes. The largest peaks in the volatility can be seen in 2022, reflecting the uncertainty expected over the announcements regarding the takeover. Investors reacted to the news rapidly, creating the large price swings.

Figure 4 shows the daily trading volume across the full period, measured in millions of shares traded. Trading volume is a useful indicator on interest in the market, often reflecting significant events surrounding a company. If something significant happens, more investors tend to buy or sell, pushing volume up.

In [ ]:
#Figure 4: Trading volume over time
plt.figure(figsize=(14, 6))
plt.bar(df['Date'], df['Volume'] / 1000000, color='steelblue', width=1.5, label='Daily Volume') #divide volume by a million to be able to have number of shares on y axis.
plt.title('Twitter Trading Volume (2013–2022)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Volume (millions of shares traded)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

The pattern in trading volume looks like that of the volatility. They are both relatively stable, with spikes at similar periods. Their relationship is explored in the scatterplot below. Noticeably, 2022 has a huge spike around the mid-year mark, around the dates Musk announced he had started to amass a large amount of Twitter stock, and the conversation for him to buy the company made it to the media. This consistently demonstrates the idea that the Musk events drew significant market participation.

Figure 5 shows a scatterplot plotting the relationship between trading volume and next-day volatility, with trading volume on the x axis and next-day volatility on the y axis. If they have a positive relationship i.e. higher volume days are followed by more volatile days, we should expect to see the points moving upward in a 45-degree angle as we move along the x axis.

In [ ]:
#Figure 5: Volume vs Next-Day Volatility
plt.figure(figsize=(10, 6))
plt.scatter(df['Volume'] / 1000000, df['Next_Day_Volatility'], 
            alpha=0.4, color='darkorange', edgecolors='none', s=15)
plt.title('Trading Volume vs Next-Day Volatility (2013–2022)', fontsize=16)
plt.xlabel('Trading Volume (millions of shares traded)')
plt.ylabel('Next-Day Volatility (%)')
plt.tight_layout()
plt.show()

The scatterplot shows a weak positive relationship between the two. The cluster of points on the left indicate that majority of trading days, volume is moderate and next-day volatility is low. As points move towards the extremities, they reflect the unusual days, with some days having very high volatility but low volume. The points on the far-right reflect those days in 2022, where trading volume was the highest throughout the history of Twitter. It is misleading however because there is a day lag between trading volume and next-day volatility. A high next-day volatility is not reflected by a large number of shares traded that same day, but the day before, explaining why the relationship is not immediately obvious. The regression analysis coming will confirm the relationship and prove that it is statistically significant.

Figure 6 shows two box plots, comparing the distribution of daily volatility between the Musk era (31st January 2022 onwards when he first purchased stock) and pre-Musk era (anytime before 31st January 2022). The boxplot shows the interquartile range, median and outliers. 

In [ ]:
#Figure 6: Volatility: Musk Era vs Non-Musk Era
fig, ax = plt.subplots(figsize=(8, 6))

musk_data = df[df['Musk_Era'] == 1]['Volatility']
non_musk_data = df[df['Musk_Era'] == 0]['Volatility']
ax.boxplot([non_musk_data, musk_data], tick_labels=['Non-Musk Era (2013–2021)', 'Musk Era (2022)'],
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', color='steelblue', alpha=0.6),
           medianprops=dict(color='red', linewidth=2))

ax.set_title('Volatility: Musk Era vs Non-Musk Era', fontsize=16)
ax.set_ylabel('Absolute Daily Return (%)')
plt.tight_layout()
plt.show()

There is a very visible difference between the two box plots. The Musk era has a noticeably higher median volatility and a much wider interquartile range, with many more extreme outliers. This confirms what the earlier charts showed; the 2022 period were significantly more volatile than the previous years. There are more data points for the non-Musk era, but this is simply due to there being a longer time for that data. Whether the differences in volatility were caused by Musk's actions directly will be unpacked in the regression that is to come.

To explore the question further, three OLS (ordinary least squares) regressions were run to see whether trading volume or the Musk events could predict next-day volatility. A regression is a statistical model that estimates the relationship between one or more predictor variables and an outcome variable. It is expected that if markets do react to the Musk events, there should be high-volume days followed by volatile days. Two Musk events were chosen for the regression, those being the day that Musk announced his intent to purchase Twitter as a whole, and the day the deal is suspended. These two events were chosen over the others because they appear to be the most significant events of the takeover period. By looking at the figure that focuses on Twitter's stock price in 2022, there are drastic changes in the price after these events. The buyout offer was the first definitive sign that Musk intended to take Twitter private, whilst the deal suspension created ambiguity, and left investors uncertain on market movement. The remaining events are still significant but were not market drivers and caused an uproar in the media like the other two.

Initially, a linear regression is run between trading volume and next-day volatility to act as a baseline for a comparison. The second regression adds two events as dummy variables to test whether these events add any additional explanation to the next-day volatility. Each event is taken across a one-week window to reflect that market reactions to announcements can have a delayed effect. A third was ran to isolate the Musk announcements, to see if they had any independent predictive power over next-day volatility.

In [ ]:
#Baseline regression using volume only.
x1 = df[['Volume']]
x1 = sm.add_constant(x1)
y1 = df['Next_Day_Volatility']
regress1 = sm.OLS(y1, x1).fit()
print(regress1.summary())

The baseline regression confirms that trading volume is statistically significant predictor of next-day volatility as it has a low p-value, less than 0.0001. For something to be statistically significant, it needs a p value less than 0.05. The positive coefficient suggests a higher trading volume is associated with greater volatility the next day, matching what the scatterplot suggested visually. The R-squared of 0.026 means the model explains around 2.6% of variation in next-day volatility, which appears low but is common for financial data. Stock prices are difficult to predict due to the vast number of factors that affect the markets. The F statistic also confirms that the model is statistically significant and is not just capturing random noise.

In [ ]:
#Regression with the Musk events, with the one-week window after it.
df['Buyout_Offer'] = ((df['Date'] >= '2022-04-14') & (df['Date'] <= '2022-04-21')).astype(int)
df['Deal_Suspended'] = ((df['Date'] >= '2022-05-13') & (df['Date'] <= '2022-05-20')).astype(int)

x2 = df[['Volume','Buyout_Offer','Deal_Suspended']]
x2 = sm.add_constant(x2)
y2 = df['Next_Day_Volatility']
regress2 = sm.OLS(y2, x2).fit()
print(regress2.summary())

Adding the two events into the regression, the numbers have changed but not by much. There is almost no change in the R-squared value, from 0.026 to 0.027, suggesting the events have very little explanatory power compared to what volume already captures. From the p values, neither the buyout offer (p = 0.328) nor the deal suspension (p = 0.646) is statistically significant, with both being above the threshold of 0.05. Volume remains significant meaning its effect remains there regardless of financial events. 

This does not mean the events are unimportant. Instead, it suggests a much more indirect effect on volatility. Announcements affect trading volume, increasing it and in turn affecting next-day volatility. They may not be an independent cause of predicting volatility but certainly play a part. 

It is worth noting that are the bottom of the regression, there is a note suggesting that there is strong multicollinearility issue, indicated by the large condition number. Multilcollinearity occurs when predictor variables are correlated with one another, making it hard for the regression model to isolate the individual effect of each variable. In this model, it is likely that volume and the announcements are correlated, as they were the days that triggered the largest spikes in trading activity. This means the model struggles to distinguish between the effect of volume and of the events themselves, explaining why the events seem to be statistically insignificant. This in turn is a limitation of the analysis, but it also reinforces the idea that volume and corporate events affect each other rather than acting as independent influences on volatility.

To test whether the events had any independent effect on volatility, a third regression was run, using only the two event dummy variables as predictors, removing volume. If the events are significant here but not in the full model, this would be consistent with multicollinearity suppressing their effect rather than the events being unimportant.

In [ ]:
#Regression with event dummies only, to test for multicollinearity.
x3_events = df[['Buyout_Offer', 'Deal_Suspended']]
x3_events = sm.add_constant(x3_events)
regress3 = sm.OLS(y2, x3_events).fit()
print(regress3.summary())

Neither the buyout offer or the deal suspension are statistically significant as their p values are 0.217 and 0.318 respectively, over the threshold of 0.05, with an R-squared of 0.001. This confirms the conclusion that the events have no independent predictive power over next-day volatility. The note suggesting there is a multicollinearity issue in the previous regression no longer appears, signifiying that the original findings stand, volume is the primary driver of next-day volatility, and that corporate events influence the trading volume, in turn affecting volatility.

As a second analytical approach, a random forest model was built using the same three predictor variables as the second regression. A random forest works by construction a large number of decision trees, with each of them learning a slightly different pattern from the data, and then averaging their predictions. Unlike an OLS regression, it does not assume a linear relationship between the predictors and the outcome. To properly evaluate the model, the data was split into 80/20 split, a training set used to build the model, and a test set to assess how well it performs on data it has not seen before.

In [ ]:
#Setting up the features and target variable of the random forest, using the same predictors as the 2nd regression.
features = ['Volume', 'Buyout_Offer', 'Deal_Suspended']
x3 = df[features]
y3 = df['Next_Day_Volatility']

#Splitting data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(x3, y3, test_size=0.2, random_state=42) #fixed random state to 42, to prevent different results every tine and allow replicability.

#Building the random forest model using 100 decision trees.
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

#Training the model on the training data.
rf_model.fit(X_train, y_train)

#Making predictions on the test set.
y_pred = rf_model.predict(X_test)

#Measuring accuracy of the predictions.
print('R-squared:', round(r2_score(y_test, y_pred), 3))
print('Mean Squared Error:', round(mean_squared_error(y_test, y_pred), 3))

#Ranking which variables contributed the most to the predictions. 
importances = pd.Series(rf_model.feature_importances_, index=features)
print('\nFeature Importances:')
print(importances.sort_values(ascending=False))

The random forest model produced an R-squared of -0.451 on the test set, indicating it performed worse than simply predicting the average next-day volatility. A negative R-squared suggests the model overfitted, a problem where the model learns the training data too precisely and it loses the ability to make accurate predictions on new data. This is a common problem when applying machine learning models to financial data, as they are noisy and difficult to predict, misleading the model (Hastie et al., 2009, pp. 587–604). A more elaborate random forest model, by adding in more variables such as lagged volatility or flagging the whole of the Musk era as a feature, could eliminate noise and be a better test.

Despite this, the random forest is still informative. Volume account for around 99.7% of the model's prediction, with the buyout offer and deal suspension contributing just 0.1% and 0.3% respectively. This is consistent with the data from the regression and reinforcing the conclusion that trading volume is the main predictor of next-day volatility, with the corporate events adding little explanatory power once volume is accounted for.

At the beginning of the blog, we set out to answer the question: how does trading volume and corporate events affect and predict next-day stock volatility in Twitter/X stock during 2013-2022?

The analysis methods of regressions, random forests and even the plots gathered from the data clearly showed Musk had some level of effect on Twitter. The price history, volatility and volume charts showed the same pattern, with relatively stable market behaviour, except price history between 2016-2018, followed by periods of spiking trading volume and volatility during the Musk takeover. The boxplot helped to confirm this, with more extreme outliers during the Musk-era.

The regression analysis showed trading volume to be the only statistically significant predictor of next-day volatility with a positive coefficient indicating higher volume today predicts greater volatility tomorrow. The two Musk events that were tested were not statistically significant after volume was controlled for, with little change to the R-squared value. A potential multicollinearity issue was flagged due to trading volume and announcements likely being correlated. It does not invalidate the findings, but instead they should be interpreted with caution. 

The random forest supported the regression findings despite producing a negative R-squared, a common problem with applying machine learning to financial datasets, due to the noise associated with them. The random forest was informative in telling us the feature importance. Volume was the most important feature in predicting next-day volatility, a conclusion that was taken from the regressions. 

To conclude, the evidence suggests that corporate events influenced next-day volatility indirectly, whereas trading volume had a more direct impact. The announcements drove up trading activity, and it was the increase in volume that predicted the volatility, with volume acting as the channel for corporate events. To develop the findings of this blog, more variables could be added to reduce the noise and improve prediction of stock prices, or the same question can be done for other stocks that have had a similar history to Twitter.

Bibliography:

Birath, S. (2022, December). From Public to Private: A Mogul’s Acquisition of Twitter – Harvard Business Law Review (HBLR). Harvard.edu. https://journals.law.harvard.edu/hblr/from-public-to-private-a-moguls-acquisition-of-twitter/

Gordon, E. (2022, October 27). Elon Musk takes Twitter private – here’s what that means for the company and its chances of success. The Conversation. https://theconversation.com/elon-musk-takes-twitter-private-heres-what-that-means-for-the-company-and-its-chances-of-success-192799

Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning. In Springer Series in Statistics (pp. 587–604). Springer New York. https://doi.org/10.1007/978-0-387-84858-7

Overfitting In Random Forests. (2026). Meegle.com. https://www.meegle.com/en_us/topics/overfitting/overfitting-in-random-forests

Wile, R. (2022, November 17). A Timeline of Elon Musk’s Takeover of Twitter. NBC News. https://www.nbcnews.com/business/business-news/twitter-elon-musk-timeline-what-happened-so-far-rcna57532